# บททดสอบ NLP (ระดับ Medium): AI นักคัดแยกข่าว

📜 สตอรี่ (Scenario): สำนักพิมพ์ของคุณได้รับพาดหัวข่าวส่งเข้ามาจำนวนมากในแต่ละวัน บรรณาธิการต้องการให้คุณ (AI Engineer) สร้างบอทที่อ่าน "พาดหัวข่าว" แล้วจัดหมวดหมู่ให้โดยอัตโนมัติ ว่าข่าวนั้นควรไปอยู่หน้าไหนของเว็บไซต์

🎯 เป้าหมายของคุณ (Your Mission): สร้าง AI (Pipeline) ที่รับข้อความพาดหัวข่าวภาษาไทย แล้วสามารถทายผลลัพธ์ออกมาเป็น 3 หมวดหมู่ (Multi-class Classification) ได้แก่:

politics (การเมือง)
sports (กีฬา)
entertainment (บันเทิง)

📌 สิ่งที่คุณต้องส่ง (Deliverables): เมื่อคุณเขียนโค้ด Train และ Test เสร็จแล้ว ให้ก๊อปปี้ 2 สิ่งนี้มาแปะส่งผมครับ:

ใบรายงานผล (Classification Report): ที่โชว์ความแม่นยำ (Accuracy, Precision, Recall) ของทั้ง 3 หมวดหมู่
ผลการทดสอบใช้งานจริง (Inference): ลองพิมพ์พาดหัวข่าวแต่งเอง (ที่ไม่มีในชุดข้อมูล) เข้าไปใน Pipeline แล้วดูว่า AI ทายว่าอะไร (เช่น ai_pipeline.predict(["ส.ส. เปิดศึกฟาดฝีปากกลางสภา"]))

In [1]:
import pandas as pd

# ชุดข้อมูลพาดหัวข่าว 50 ข้อความ
data = {
    'text': [
        # กีฬา (Sports)
        "แมนยูบุกเฉือนชนะลิเวอร์พูล 1-0 ท้ายเกม", "วอลเลย์บอลหญิงไทย ทะลุรอบ 8 ทีมสุดท้าย",
        "นักเตะเจ็บหนัก พักยาวทั้งฤดูกาล", "โค้ชประกาศลาออก หลังทำผลงานแย่ต่อเนื่อง",
        "ทีมชาติไทยเตรียมอุ่นเครื่องกับญี่ปุ่นเดือนหน้า", "เทนนิสระดับโลกนาดาลคว้าแชมป์แกรนด์สแลม",
        "แฟนบอลแห่ซื้อตั๋วเกมนัดชิงเอฟเอคัพจนเว็บล่ม", "ดราม่าวงการมวย กรรมการตัดสินค้านสายตา",
        "เปิดสถิติ กองหน้าดาวรุ่งยิงทะลุ 20 ประตู", "กอล์ฟหญิงโปรโมคว้าแชมป์ที่อเมริกา",
        "นักบาสเอ็นบีเอทุบสถิติทำแต้มสูงสุดในประวัติศาสตร์", "ฟุตซอลไทยฟอร์มดุ ถล่มเวียดนามยับเยิน",
        "แบดมินตันวิวสุดยอด คว้าเหรียญทองชิงแชมป์โลก", "สมาคมฟุตบอลประกาศแต่งตั้งผู้จัดการทีมคนใหม่",
        "เปิดโผ 23 แข้งลุยศึกฟุตบอลโลกโควต้าเอเชีย", "แชมป์เก่าร่วงตกรอบแรกแบดมินตันโอลิมปิก",
        "นักเตะดังย้ายทีมค่าตัวสถิติโลกทะลุร้อยล้านยูโร",

        # บันเทิง (Entertainment)
        "ดาราสาวประกาศแต่งงานสายฟ้าแลบ แฟนคลับช็อก", "ซีรีส์เกาหลีเรื่องใหม่ เรตติ้งพุ่งอันดับหนึ่ง",
        "เปิดใจนักร้องดัง หลังเจอเจอมรสุมชีวิต", "คอนเสิร์ตวงร็อคบัตรขายเกลี้ยงใน 5 นาที",
        "ดราม่าเดือด แฮชแท็กแบนพระเอกดังขึ้นเทรนด์ทวิต", "ผู้กำกับรางวัลออสการ์เตรียมสร้างหนังฟอร์มยักษ์",
        "ส่องแฟชั่นพรมแดง งานประกาศรางวัลคนบันเทิง", "ชาวเน็ตจับตา คู่จิ้นนอกจอแอบควงกันไปเที่ยว",
        "หนังไทยกวาดรายได้ทะลุ 100 ล้านบาททั่วประเทศ", "รายการเซอร์ไววัลเปิดตัวผู้เข้าแข่งขันซีซั่นใหม่",
        "นักแสดงหน้าใหม่แจ้งเกิดจากบทนางร้าย", "หลุดภาพเบื้องหลัง กองถ่ายละครฟอร์มใหญ่ช่องดัง",
        "ศิลปินเคป็อปเตรียมคัมแบ็ค ปล่อยทีเซอร์ใหม่", "นางงามเวทีดังชี้แจงประเด็นดราม่าเรื่องชุด",
        "นักร้องหนุ่มปล่อยเพลงใหม่ ยอดวิวทะลุล้านในวันเดียว", "ชาวเน็ตแห่ชื่นชม นางเอกสาวบริจาคเงินช่วยเหลือเด็ก",
        "ปิดฉากความรัก 5 ปี ดาราคู่รักประกาศเลิกทางกัน",

        # การเมือง (Politics)
        "สภาถกเดือด ร่างพ.ร.บ.งบประมาณปีหน้า", "นายกฯ ลงพื้นที่ตรวจน้ำท่วม พร้อมเยียวยาประชาชน",
        "พรรคฝ่ายค้านเตรียมยื่นอภิปรายไม่ไว้วางใจ", "ม็อบรวมตัวประท้วงหน้าทำเนียบ ร้องแก้ปัญหาปากท้อง",
        "กกต.ประกาศวันเลือกตั้งซ่อมอย่างเป็นทางการ", "จับตาการปรับครม.ครั้งใหญ่ รัฐมนตรีหลุดเก้าอี้เพียบ",
        "นักการเมืองดังถูกป.ป.ช.ชี้มูลความผิดคอร์รัปชัน", "โพลเผย คะแนนนิยมรัฐบาลลดลงอย่างต่อเนื่อง",
        "อดีตนายกฯ โพสต์เฟซบุ๊กวิจารณ์การแก้ปัญหาน้ำ", "ร่างกฎหมายสมรสเท่าเทียมผ่านวาระแรกแล้ว",
        "พรรคการเมืองเปิดตัวแคนดิเดตนายกฯ คนใหม่", "ส.ส.ลาออกย้ายพรรค เตรียมรับการเลือกตั้งครั้งหน้า",
        "รัฐบาลอนุมัติโครงการเยียวยาแจกเงินประชาชน", "ศาลรัฐธรรมนูญนัดอ่านคำวินิจฉัยคดียุบพรรค",
        "ถกเดือดประเด็นแก้รัฐธรรมนูญ ส.ว.ค้านหัวชนฝา", "กระทรวงพาณิชย์สั่งคุมราคาสินค้า ป้องกันฉวยโอกาส"
    ],
    'label': [
        'sports','sports','sports','sports','sports','sports','sports','sports','sports','sports','sports','sports','sports','sports','sports','sports','sports',
        'entertainment','entertainment','entertainment','entertainment','entertainment','entertainment','entertainment','entertainment','entertainment','entertainment','entertainment','entertainment','entertainment','entertainment','entertainment','entertainment','entertainment',
        'politics','politics','politics','politics','politics','politics','politics','politics','politics','politics','politics','politics','politics','politics','politics','politics'
    ]
}

# สร้างตาราง สลับข้อมูลให้คละกัน (Shuffle) และเซฟเป็น CSV
df = pd.DataFrame(data)
# print('-'*20)
# display(df)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
# print('-'*20)
# display(df)
df.to_csv('news_data.csv', index=False)
print(f"✅ สร้างไฟล์ news_data.csv จำนวน {len(df)} บรรทัด สำเร็จแล้ว!")

✅ สร้างไฟล์ news_data.csv จำนวน 50 บรรทัด สำเร็จแล้ว!


## Data Preprocessing การเตรียมและจัดการข้อมูล

In [2]:
from sklearn.model_selection import train_test_split

# แบ่งข้อมูลตาม Column
X = df['text']
Y = df['label']

In [3]:
# แบ่งข้อมูล train และ test
X_train,X_test,Y_train,Y_test = train_test_split(X, Y, random_state = 42, test_size = 0.25)

# print(X_train)
# ถ้าเรากำหนดเลขแบบ [4] แบบนี้ไปตรงๆจะเป็นการไปหาว่ามีเลขกำกับข้อมูลที่ 4 อยู่ในนี้มั้ย
# แต่ถ้าอยากเอาข้อมูลอันดับที่ 4 ในข้อมูลของ pandas ที่มี tuple เป็นข้อมูลด้านในให้ใช้
print(X_train.iloc[4])


อดีตนายกฯ โพสต์เฟซบุ๊กวิจารณ์การแก้ปัญหาน้ำ


In [4]:
!pip install pythainlp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.9/21.9 MB 31.1 MB/s eta 0:00:00


In [5]:
"""
สร้าง Function การจัดการข้อมูลตัวอักษรนั้นต้องจัดการข้อมูลโดยที่
1.จัดการตัวอักษรพิเศษให้เป็นช่องว่าง
2.ถ้าเป็นภาษาไทยที่ไม่เน้นการเว้นวรรคเป็นหลัก ต้องแบ่งเป็นคำ ๆ หรือการทำ word_tokenize ด้วยการเปิดดูพจนานุกรม
3.จัดการพวกคำเชื่อมที่ไม่จำเป็นออก
"""

from pythainlp import word_tokenize
from pythainlp.corpus import thai_stopwords
import re

# คำเชื่อมที่เราไม่ต้องการ
stopwords = list(thai_stopwords())
# print(stopwords)

def clean_thai_texts(text):
  # ทำให้อักขระพิเศษเป็นช่องว่า่ง
  text = re.sub(r'[^ก-๙a-zA-Z\s]', '', text)

  # แบ่งคำด้วยการเว้นวรรคโดยอ้างอิงจากพจนานุกรม(newmm)
  tokens = word_tokenize(text, engine = 'newmm' )

  # จัดการคำเชื่อมที่ไม่จำเป็น
  # ตรวจสอบคำในข้อความว่ามีคำเชื่อมมั้ย ถ้ามีก็เอาออก ถ้าไม่มีก็เหมือนเดิม และ ในตอนสุดท้ายข้อความต้องไม่ใช่ข้อความว่าง
  cleaned_tokens = [word for word in tokens if word not in stopwords and word.strip() != '']

  # ทำให้ระหว่างคำทุกคำมีเว้นวรรคหรือช่องว่าง และกลายเป็น Str เพื่อไปใช้ในการทำ model
  return ' '.join(cleaned_tokens)

In [6]:
# จัดการทำความสะอาดโดยใช้ Function ที่สร้างไว้
X_train_clean = X_train.apply(clean_thai_texts)
X_test_clean = X_test.apply(clean_thai_texts)

print(type(X_train_clean))
display(X_train_clean)

<class 'pandas.core.series.Series'>


,text
8,เน็ต แห่ ชื่นชม นางเอก สาว บริจาค เงินช่วยเหลื...
3,สส ลาออก ย้าย พรรค เตรียม การเลือกตั้ง ครั้งหน้า
6,รายการ เซอร์ ไว วัล เปิดตัว ผู้เข้าแข่งขัน ซีซั่น
41,ผู้กำกับ รางวัล ออสการ์ เตรียม สร้าง หนัง ฟอร์...
46,นายกฯ โพสต์ เฟซบุ๊ก วิจารณ์ การแก้ปัญหา น้ำ
47,โผ แข้ง ลุย ศึก ฟุตบอล โลก โควต้า เอเชีย
15,แฟน บอล แห่ ซื้อ ตั๋ว เกม นัด ชิง เอฟเอคัพ เว็...
9,เปิดใจ นักร้อง เจอ เจอ มรสุม ชีวิต
16,โพล เผย คะแนนนิยม รัฐบาล ลดลง ต่อเนื่อง
24,นักร้อง หนุ่ม ปล่อย เพลง ยอด วิว ทะลุ ล้าน


## Features Extraction การแปลงอักขระเป็นรูปแบบตัวเลขหรือเวกเตอร์ (word embedding)

In [7]:
# Import สิ่งที่จะใช้แปลงข้อความเป็นตัวเลข
from sklearn.feature_extraction.text import TfidfVectorizer

# สร้างเครื่องแปลงข้อความเป็นตัวเลขขึ้นมา
"""
คำสั่ง ngram_range=(1, 2) จะทำให้ระบบเก็บทั้งคำเดี่ยวๆ และคำคู่กัน (เช่น "นักเตะ", "เจ็บหนัก", "นักเตะเจ็บหนัก")
ส่งไปให้ LogisticRegression ครับ วิธีนี้ดึงศักยภาพของ Scikit-learn ออกมาได้สูงสุด
และเหมาะกับข้อมูลน้อยๆ ของเรามากที่สุดครับ!
"""
tfid = TfidfVectorizer(ngram_range = (1,2))

# ให้ แปลงเป็นตัวเลขไปพร้อมกับการเรียนรู้คำศัพท์เลย
X_train_vector = tfid.fit_transform(X_train_clean)

# แปลงข้อสอบเป็นตัวเลข
X_test_vector = tfid.transform(X_test_clean)

# ตรวจสอบข้อมูลที่แปลง
print(X_train_vector.toarray())


[[0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.26100299 0.26100299]
 ...
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]


## Predictive Modeling การจัดทำ Model การคาดการณ์

In [8]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth = 4, random_state = 42, criterion = "gini")

# นี่คือการ Training Model
# ใช้คำสั่ง .fit() โดยใช้ โจทย์เลข (X_train_vector) และ เฉลย (y_train)
model.fit(X_train_vector,Y_train)

DecisionTreeClassifier(max_depth=4, random_state=42)

## Model Evaluation ผลลัพธ์ของโมเดล


In [9]:
# นำเข้าข้อมูลมาตรวจสอบ (การให้คะแนน)
from sklearn.metrics import classification_report, accuracy_score

Y_predict = model.predict(X_test_vector)

print(accuracy_score(Y_predict,Y_test)*100)
print(classification_report(Y_test, Y_predict))

23.076923076923077
               precision    recall  f1-score   support

entertainment       0.18      0.67      0.29         3
     politics       0.50      0.17      0.25         6
       sports       0.00      0.00      0.00         4

     accuracy                           0.23        13
    macro avg       0.23      0.28      0.18        13
 weighted avg       0.27      0.23      0.18        13



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Build Pipeline

In [17]:
from sklearn.pipeline import Pipeline

# 1. 🛠️ สร้างท่อ (Pipeline) รวบยอด
ai_pipeline = Pipeline([
    # ท่อท่อนที่ 1: เอาข้อความเข้าเครื่อง Tfidf (โดยแอบสั่งให้มันไปเรียกโรงงาน clean_thai_text ก่อนเสมอ)
    ('vectorizer', TfidfVectorizer(preprocessor = clean_thai_texts, ngram_range = (1,2))),
    # ท่อท่อนที่ 2: ส่งตัวเลขที่แปลงเสร็จแล้วเข้าสมองกล
    ('classifier', DecisionTreeClassifier(max_depth = 4, random_state = 42, criterion = "gini"))
])

# 2. 🧠 สั่งให้ Pipeline เรียนรู้จากข้อมูลดิบๆ (X_train)
# ตอนนี้เราไม่ต้องห่วงเรื่องการแปลงข้อมูลก่อนแล้ว Pipeline จะจัดการให้หมดหลังบ้าน
ai_pipeline.fit(X_train,Y_train)

pipeline_predict = ai_pipeline.predict(X_test)
print(f"ความแม่นยำใหม่: {accuracy_score(Y_test, pipeline_predict)*100:.2f}%\n")
print(classification_report(Y_test, pipeline_predict))
# 4. 🎯 ทดสอบส่งข่าวจริงที่แต่งเองเข้าไปทายผล!
test_news = [
    "บุกสภา! พรรคฝ่ายค้านงัดหลักฐานเด็ดซักฟอกนายก",
    "ลิเวอร์พูลฟอร์มโหด คว้าแชมป์ยูฟ่าสมัยที่ 7",
    "ดาราสาวเปิดตัวแฟนหนุ่มกลางงานคอนเสิร์ต"
]
predictions = ai_pipeline.predict(test_news)
for news, cat in zip(test_news, predictions):
    print(f"ข่าว: {news} ➡️ หมวด: {cat}")

ความแม่นยำใหม่: 23.08%

               precision    recall  f1-score   support

entertainment       0.18      0.67      0.29         3
     politics       0.50      0.17      0.25         6
       sports       0.00      0.00      0.00         4

     accuracy                           0.23        13
    macro avg       0.23      0.28      0.18        13
 weighted avg       0.27      0.23      0.18        13

ข่าว: บุกสภา! พรรคฝ่ายค้านงัดหลักฐานเด็ดซักฟอกนายก ➡️ หมวด: entertainment
ข่าว: ลิเวอร์พูลฟอร์มโหด คว้าแชมป์ยูฟ่าสมัยที่ 7 ➡️ หมวด: sports
ข่าว: ดาราสาวเปิดตัวแฟนหนุ่มกลางงานคอนเสิร์ต ➡️ หมวด: entertainment


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
